# Week 8 — Hyperparameter Tuning and Robust Evaluation

**Course context:** Every model this course has used so far has hyperparameters (n_estimators, max_depth, learning_rate...) mostly left at their defaults, or briefly tuned with a small grid. This week makes hyperparameter tuning a first-class skill: manual tuning, GridSearchCV, RandomizedSearchCV, and Optuna (a much smarter search strategy) — plus K-Fold and Stratified K-Fold for evaluating the results reliably.

**Data files needed (same folder as this notebook):** `Telco-Customer-Churn-Full.csv` (Day 7) — the full Kaggle-style Telco dataset, different/richer than the simplified one used in Weeks 5 and 7.

**A note on runtime:** several exercises this week (Optuna with 50 trials, large grid searches) are trimmed down from the original source material to keep this notebook running in a reasonable time. The full-size versions are noted in comments — expand them if you have time to spare, the tuning *logic* is identical either way.

**How this notebook is organized:** one section per day, each with: what it covers → why it matters for AI work → code → pitfalls.


## Day 1 — Manual Hyperparameter Tuning: Feel the Effect Before Automating It

**What it covers:** Comparing a default Random Forest against a manually-adjusted one, to build direct intuition for what changing `n_estimators` and `max_depth` actually does to performance.

**Why it matters for AI work:** Before reaching for automated search (Day 2+), it's worth manually trying a couple of configurations and watching the numbers move. This builds intuition that makes the automated search results easier to interpret later — you'll recognize *why* the search landed where it did, instead of treating it as a black box.

**What each hyperparameter tends to do:**
- More trees (`n_estimators`) — generally more stable predictions, with diminishing returns and higher computation cost past a point
- Limiting `max_depth` — trades some fitting flexibility for less overfitting risk; whether this helps or hurts depends on whether the default was already overfitting or underfitting on your specific data

**Pitfalls:**
- A "tuned" model isn't automatically better — sometimes manual tuning makes things worse if you're guessing rather than searching systematically. That's precisely the motivation for Day 2's automated approaches.
- Comparing exactly two configurations (default vs one manual guess) tells you very little about the broader hyperparameter landscape — it's a starting intuition, not a real search.


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_default = RandomForestClassifier(random_state=42)
rf_default.fit(X_train, y_train)
y_predict_default = rf_default.predict(X_test)
accuracy_default = accuracy_score(y_test, y_predict_default)

print(f"Default Model Accuracy: {accuracy_default:.4f}")
print("\nClassification Report (default):\n", classification_report(y_test, y_predict_default))

Default Model Accuracy: 0.9649

Classification Report (default):
               precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [2]:
rf_tuned = RandomForestClassifier(n_estimators=400, max_depth=5, random_state=42)
rf_tuned.fit(X_train, y_train)
y_pred_tuned = rf_tuned.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"Manually Tuned Model Accuracy: {accuracy_tuned:.4f}")
print("\nClassification Report (tuned):\n", classification_report(y_test, y_pred_tuned))

print(f"\nDifference vs default: {accuracy_tuned - accuracy_default:+.4f}")
print("On an already-easy dataset like this one, the gap from manual tuning alone is often small -")
print("automated search (next) matters more on harder, messier real-world datasets.")

Manually Tuned Model Accuracy: 0.9649

Classification Report (tuned):
               precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


Difference vs default: +0.0000
On an already-easy dataset like this one, the gap from manual tuning alone is often small -
automated search (next) matters more on harder, messier real-world datasets.


## Day 2 — GridSearchCV vs RandomizedSearchCV

**What it covers:** The two most common automated hyperparameter search strategies in scikit-learn, compared directly on the same problem.

**Why it matters for AI work:** Grid search is exhaustive (tries every combination) — thorough but expensive, and the cost explodes combinatorially as you add more hyperparameters or more values per hyperparameter. Randomized search samples a fixed number of *random* combinations instead — usually finds a similarly good result much faster, especially when some hyperparameters matter much more than others (common in practice).

**When to use what:**
- **GridSearchCV** — small search spaces (few hyperparameters, few values each), or when you need the guarantee of trying every combination
- **RandomizedSearchCV** — larger search spaces, limited compute/time budget. `n_iter` controls how many random combinations to try — more iterations = better chance of finding a good combination, more compute cost.

**Pitfalls:**
- Grid search's cost is the product of every parameter's value count, times `cv` folds. `3 x 3 x 3 = 27` combinations x 5 folds = 135 model fits — adding just one more 3-value hyperparameter makes it `3^4 x 5 = 405` fits. This multiplies fast, which is exactly why randomized search (and Day 3's Optuna) exist.
- Randomized search isn't guaranteed to find the true best combination (it's sampling, not exhaustive) — but in practice, with enough iterations, it usually gets close, for a fraction of the compute.


In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import numpy as np

data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

best_grid_model = grid_search.best_estimator_
accuracy_grid = accuracy_score(y_test, best_grid_model.predict(X_test))

print(f"Best Hyperparameters (Grid Search): {grid_search.best_params_}")
print(f"Grid Search Test Accuracy: {accuracy_grid:.4f}")

Best Hyperparameters (Grid Search): {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 150}
Grid Search Test Accuracy: 1.0000


In [4]:
param_dist = {
    "n_estimators": np.arange(50, 200, 10),
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10, 20],
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
)
random_search.fit(X_train, y_train)

best_random_model = random_search.best_estimator_
accuracy_random = accuracy_score(y_test, best_random_model.predict(X_test))

print(f"Best Hyperparameters (Random Search): {random_search.best_params_}")
print(f"Random Search Test Accuracy: {accuracy_random:.4f}")

print(f"\nGrid Search tried {3*3*3} combinations, Random Search tried only 20 - and often lands close.")

Best Hyperparameters (Random Search): {'n_estimators': np.int64(140), 'min_samples_split': 5, 'max_depth': None}
Random Search Test Accuracy: 1.0000

Grid Search tried 27 combinations, Random Search tried only 20 - and often lands close.


## Day 3 — Optuna: Smarter Hyperparameter Search

**What it covers:** Optuna, a library that searches the hyperparameter space **intelligently** rather than randomly or exhaustively — it uses the results of previous trials to decide where to search next.

**Why it matters for AI work:** Optuna (and similar "Bayesian optimization" tools) is the modern standard for serious hyperparameter tuning, especially for expensive-to-train models (deep learning, large boosting models) where you can only afford a limited number of training runs. It tends to find better configurations than random search using far fewer trials.

**How it differs from Grid/Random search:**
- **Grid search** — exhaustive, no learning between trials
- **Random search** — random, no learning between trials
- **Optuna** — after each trial, it updates its internal belief about which regions of the search space look promising, and focuses future trials there. This is why it can be more efficient with a comparable number of trials.

**Pitfalls:**
- `trial.suggest_float(...)`/`suggest_int(...)` define a *range* to search within — setting these ranges too narrow can exclude the actual best value; too wide wastes trials exploring clearly bad regions. Some judgment (from Day 1's manual intuition) helps set sensible ranges.
- Optuna's advantage over random search shows up more clearly with **more trials and more hyperparameters** than this trimmed example uses — on simple, low-dimensional problems the gap can be small.


In [5]:
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)   # quiet the trial-by-trial log spam

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

baseline_model = XGBClassifier(eval_metric="logloss", random_state=42)
baseline_model.fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, baseline_model.predict(X_test))
print(f"Baseline XGBoost Accuracy: {baseline_accuracy:.4f}")

Baseline XGBoost Accuracy: 0.9649


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    model = XGBClassifier(eval_metric="logloss", random_state=42, **params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

# Trimmed to 20 trials for runtime (original exercise used 50 - safe to raise this if you have time)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, show_progress_bar=False)

print("Best Hyperparameters (Optuna):", study.best_params)
print("Best Accuracy (Optuna):", study.best_value)
print(f"\nImprovement over baseline: {study.best_value - baseline_accuracy:+.4f}")

Best Hyperparameters (Optuna): {'n_estimators': 139, 'max_depth': 15, 'learning_rate': 0.16730144872419994, 'subsample': 0.7853057963108915, 'colsample_bytree': 0.8242966822191383}
Best Accuracy (Optuna): 0.9736842105263158

Improvement over baseline: +0.0088


In [7]:
# Compare against a trimmed Grid Search and Random Search on the same data, for a fair three-way comparison
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
}
grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric="logloss", random_state=42),
    param_grid=param_grid, scoring="accuracy", cv=3,
)
grid_search.fit(X_train, y_train)
print("Grid Search Best Accuracy:", grid_search.best_score_)

param_dist = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
}
random_search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric="logloss", random_state=42),
    param_distributions=param_dist, n_iter=15, scoring="accuracy", cv=3, random_state=42,
)
random_search.fit(X_train, y_train)
print("Random Search Best Accuracy:", random_search.best_score_)

print(f"\nOptuna Best Accuracy: {study.best_value:.4f}")

Grid Search Best Accuracy: 0.9670326478447775


Random Search Best Accuracy: 0.9626176368072499

Optuna Best Accuracy: 0.9737


## Day 4 — Tuning Regularization Strength: Ridge and Lasso Alpha

**What it covers:** Comparing plain Linear Regression against Ridge and Lasso at a fixed `alpha`, then inspecting the resulting coefficients to see regularization's effect directly.

**Why it matters for AI work:** `alpha` (regularization strength, introduced in Week 5 Day 3) is itself a hyperparameter that needs tuning — too high and the model underfits (coefficients pushed too close to zero), too low and you're back to plain linear regression's overfitting risk. This section makes that trade-off visible by comparing actual coefficient values.

**What to look for in the coefficients:**
- Plain Linear Regression's coefficients are the "unconstrained" baseline
- Ridge's coefficients should all be **smaller in magnitude** than the unregularized ones (shrunk toward zero, but rarely exactly zero)
- Lasso's coefficients should show some values pushed to **exactly zero** — effectively excluding those features from the model entirely

**Pitfalls:**
- A single fixed `alpha=0.1` here is illustrative, not necessarily optimal — in a real project you'd tune `alpha` itself with `GridSearchCV` (`sklearn.linear_model.RidgeCV`/`LassoCV` even have this built in as a convenience).
- Regularized linear models are scale-sensitive (Week 5 Day 3's pitfall) — for a fair comparison, features should be scaled first in a real project; this exercise skips that for simplicity of demonstrating the coefficient-shrinking effect alone.


In [8]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# NOTE: original exercise used fetch_california_housing() (needs an external host not reachable
# from every network). Using a synthetic housing-style dataset with the SAME 8 feature names,
# so the code and interpretation below still directly transfer to the real dataset.
np.random.seed(7)
n = 3000
feature_names = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"]

med_inc = np.random.gamma(shape=5, scale=1.5, size=n)
house_age = np.random.uniform(1, 52, size=n)
ave_rooms = np.random.uniform(2, 10, size=n)
ave_bedrms = ave_rooms * np.random.uniform(0.15, 0.3, size=n)
population = np.random.uniform(100, 5000, size=n)
ave_occup = np.random.uniform(1, 6, size=n)
latitude = np.random.uniform(32, 42, size=n)
longitude = np.random.uniform(-124, -114, size=n)

X = np.column_stack([med_inc, house_age, ave_rooms, ave_bedrms, population, ave_occup, latitude, longitude])
y = (0.5 * med_inc + 0.01 * house_age + 0.05 * ave_rooms - 0.1 * ave_occup
     + np.random.normal(0, 0.6, size=n))
y = np.clip(y, 0.15, 5.0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Feature Names:", feature_names)
print("\nSample Data:\n", pd.DataFrame(X, columns=feature_names).head())

Feature Names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Sample Data:
       MedInc   HouseAge  AveRooms  AveBedrms   Population  AveOccup  \
0  14.031116  18.281537  9.503984   1.654074  1339.436548  3.184292   
1   5.596138  35.320178  4.600971   0.700057  3019.749790  5.002149   
2   4.742169  25.073922  4.555541   1.191184  4165.598543  3.201327   
3   7.006695  19.102532  7.085278   1.138477   626.201813  5.414414   
4  10.842507  12.621288  3.256463   0.869198   849.850987  4.336405   

    Latitude   Longitude  
0  37.387369 -115.155755  
1  32.291005 -118.159925  
2  32.547532 -123.079311  
3  36.024994 -121.620795  
4  39.707721 -116.304895  


In [9]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
mse_lr = mean_squared_error(y_test, lr_model.predict(X_test))
print(f"Linear Regression MSE (No Regularization): {mse_lr:.4f}")
print("Coefficients:\n", dict(zip(feature_names, lr_model.coef_.round(3))))

ridge_model = Ridge(alpha=0.1)
ridge_model.fit(X_train, y_train)
mse_ridge = mean_squared_error(y_test, ridge_model.predict(X_test))
print(f"\nRidge Regression MSE: {mse_ridge:.4f}")
print("Coefficients:\n", dict(zip(feature_names, ridge_model.coef_.round(3))))

lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train, y_train)
mse_lasso = mean_squared_error(y_test, lasso_model.predict(X_test))
print(f"\nLasso Regression MSE: {mse_lasso:.4f}")
print("Coefficients:\n", dict(zip(feature_names, lasso_model.coef_.round(3))))
print("\nNotice Lasso pushes some coefficients to exactly 0 - automatic feature selection.")

Linear Regression MSE (No Regularization): 0.4364
Coefficients:
 {'MedInc': np.float64(0.309), 'HouseAge': np.float64(0.006), 'AveRooms': np.float64(0.024), 'AveBedrms': np.float64(0.088), 'Population': np.float64(0.0), 'AveOccup': np.float64(-0.089), 'Latitude': np.float64(0.006), 'Longitude': np.float64(0.001)}

Ridge Regression MSE: 0.4364
Coefficients:
 {'MedInc': np.float64(0.309), 'HouseAge': np.float64(0.006), 'AveRooms': np.float64(0.024), 'AveBedrms': np.float64(0.088), 'Population': np.float64(0.0), 'AveOccup': np.float64(-0.089), 'Latitude': np.float64(0.006), 'Longitude': np.float64(0.001)}

Lasso Regression MSE: 0.4389
Coefficients:
 {'MedInc': np.float64(0.299), 'HouseAge': np.float64(0.006), 'AveRooms': np.float64(0.026), 'AveBedrms': np.float64(0.0), 'Population': np.float64(0.0), 'AveOccup': np.float64(-0.043), 'Latitude': np.float64(0.0), 'Longitude': np.float64(0.0)}

Notice Lasso pushes some coefficients to exactly 0 - automatic feature selection.


## Day 5 — K-Fold vs Stratified K-Fold: Why Class Balance Matters in Cross-Validation

**What it covers:** Comparing plain K-Fold cross-validation against Stratified K-Fold on an imbalanced dataset.

**Why it matters for AI work:** With imbalanced data (Week 7 Day 6), a plain random K-Fold split can accidentally create folds with wildly different class balances by chance — one fold might have almost no minority-class examples, making that fold's evaluation unreliable. Stratified K-Fold fixes this by preserving the overall class proportion in every fold.

**When to use what:**
- **KFold** — fine for regression, or classification with well-balanced classes
- **StratifiedKFold** — the correct default for classification whenever there's any meaningful class imbalance (which, in practice, is most real classification datasets)

**Pitfalls:**
- The difference between KFold and StratifiedKFold is easy to miss on a *mildly* imbalanced dataset — the effect becomes much clearer and more consequential the more imbalanced the classes are (extreme cases like fraud detection, where StratifiedKFold is close to mandatory practice).


In [10]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold

# NOTE: original exercise used a real credit-card-fraud dataset from an external host
# (storage.googleapis.com) not reachable from every network - reusing the Week 7 Day 6-style
# synthetic imbalanced dataset so this notebook is self-contained.
np.random.seed(42)
n_samples = 5000
n_features = 8
fraud_rate = 0.03

X_fraud = np.random.randn(n_samples, n_features)
y_fraud = np.zeros(n_samples, dtype=int)
n_fraud = int(n_samples * fraud_rate)
fraud_idx = np.random.choice(n_samples, n_fraud, replace=False)
y_fraud[fraud_idx] = 1
X_fraud[fraud_idx] += np.random.randn(n_fraud, n_features) * 1.5 + 2

df_fraud = pd.DataFrame(X_fraud, columns=[f"V{i+1}" for i in range(n_features)])
df_fraud["Class"] = y_fraud

print("Class Distribution:\n", df_fraud["Class"].value_counts())

X = df_fraud.drop(columns=["Class"])
y = df_fraud["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Class Distribution:
 Class
0    4850
1     150
Name: count, dtype: int64


In [11]:
rf_model = RandomForestClassifier(random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring="accuracy")
print(f"K-Fold scores: {scores_kfold}")
print(f"Mean Accuracy (K-Fold): {scores_kfold.mean():.4f}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_stratified = cross_val_score(rf_model, X_train, y_train, cv=skf, scoring="accuracy")
print(f"\nStratified K-Fold scores: {scores_stratified}")
print(f"Mean Accuracy (Stratified K-Fold): {scores_stratified.mean():.4f}")

print(f"\nStd Dev - K-Fold: {scores_kfold.std():.4f}, Stratified K-Fold: {scores_stratified.std():.4f}")
print("Lower std dev with stratification usually means more CONSISTENT fold-to-fold evaluation -")
print("less luck-of-the-draw in which fold happened to get more/fewer minority-class examples.")

K-Fold scores: [0.995   0.99625 0.99625 0.99625 0.9925 ]
Mean Accuracy (K-Fold): 0.9952



Stratified K-Fold scores: [0.99875 0.995   0.99125 0.99375 0.99625]
Mean Accuracy (Stratified K-Fold): 0.9950

Std Dev - K-Fold: 0.0015, Stratified K-Fold: 0.0025
Lower std dev with stratification usually means more CONSISTENT fold-to-fold evaluation -
less luck-of-the-draw in which fold happened to get more/fewer minority-class examples.


## Day 6 — Tuning Two Very Different Model Types: Gradient Boosting and SVM

**What it covers:** GridSearchCV on Gradient Boosting (tree-based) and RandomizedSearchCV on a Support Vector Machine (SVC) — a model type not covered elsewhere in this course, included here to show that the *tuning workflow* is identical even though the underlying algorithm and hyperparameters are completely different.

**Why it matters for AI work:** The pattern — define a search space, wrap a model in Grid/RandomizedSearchCV, fit, inspect `.best_params_`/`.best_score_`, evaluate `.best_estimator_` on the test set — is universal across scikit-learn model types. Once you've internalized it, tuning a new algorithm you've never used before is mostly about knowing *which* hyperparameters exist for it, not relearning the process.

**Brief primer on SVM hyperparameters (since SVM itself isn't covered elsewhere this course):**
- `C` — regularization strength (inverse: **smaller** C = stronger regularization, simpler decision boundary; larger C = fits training data more closely, more overfitting risk)
- `kernel` — the shape of decision boundary SVM can draw (`linear` = straight line/plane; `rbf`/`poly`/`sigmoid` = various curved boundaries)
- `gamma` — how far a single training example's influence reaches (relevant for non-linear kernels)

**Pitfalls:**
- Random Search over a mix of continuous (`np.logspace`) and categorical (`kernel`) hyperparameters, as done here, is a common realistic search space shape — worth noticing this pattern for your own future tuning.


In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5, 7],
}
grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid, scoring="accuracy", cv=5, n_jobs=-1,
)
grid_search.fit(X_train, y_train)

best_grid_model = grid_search.best_estimator_
accuracy_grid = accuracy_score(y_test, best_grid_model.predict(X_test))

print(f"Best Parameters (GridSearchCV, Gradient Boosting): {grid_search.best_params_}")
print(f"Test Accuracy: {accuracy_grid:.4f}")

Best Parameters (GridSearchCV, Gradient Boosting): {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}
Test Accuracy: 1.0000


In [13]:
param_dist = {
    "C": np.logspace(-3, 3, 10),
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "gamma": ["scale", "auto"],
}
random_search = RandomizedSearchCV(
    estimator=SVC(random_state=42),
    param_distributions=param_dist, n_iter=20, scoring="accuracy", cv=5, n_jobs=-1, random_state=42,
)
random_search.fit(X_train, y_train)

best_random_model = random_search.best_estimator_
accuracy_random = accuracy_score(y_test, best_random_model.predict(X_test))

print(f"Best Parameters (RandomizedSearchCV, SVM): {random_search.best_params_}")
print(f"Test Accuracy: {accuracy_random:.4f}")

Best Parameters (RandomizedSearchCV, SVM): {'kernel': 'poly', 'gamma': 'auto', 'C': np.float64(0.021544346900318832)}
Test Accuracy: 0.9667


## Day 7 — Mini Project: Fully Tuned Churn Prediction, with Cross-Validation Confirmation

**What it covers:** The complete tuning workflow on a real, messy dataset: load -> clean -> encode -> scale -> baseline model -> RandomizedSearchCV -> evaluate the tuned model -> confirm the result holds up under cross-validation (not just a single lucky train/test split).

**Dataset:** the full Kaggle-style Telco churn dataset (`Telco-Customer-Churn-Full.csv`) — richer than the simplified version used in Weeks 5 and 7, with 19 real customer attributes (internet service type, contract terms, payment method, etc.).

**Why the final cross-validation step matters:** a single train/test split's accuracy can be somewhat lucky or unlucky depending on which rows landed in the test set. Running `cross_val_score` on the *tuned* model's configuration afterward confirms whether the improvement from tuning is real and consistent, or partly an artifact of this particular split.

**Pitfalls (carried over from earlier weeks, worth restating in a full pipeline like this):**
- `TotalCharges` needs `pd.to_numeric(..., errors="coerce")` — in the raw data it's stored as text and a handful of rows have blank/invalid values (new customers with 0 tenure) that become `NaN` after conversion and need to be filled.
- `customerID` is dropped before modeling — same reasoning as Week 5 Day 7 and Week 7 Day 7: an arbitrary ID has no genuine predictive relationship with churn.


In [14]:
df_telco = pd.read_csv("Telco-Customer-Churn-Full.csv")
df_telco.info()
print("\nClass Distribution:\n", df_telco["Churn"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [15]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

df_telco["TotalCharges"] = pd.to_numeric(df_telco["TotalCharges"], errors="coerce")
df_telco.fillna({"TotalCharges": df_telco["TotalCharges"].median()}, inplace=True)

label_encoder = LabelEncoder()
for column in df_telco.select_dtypes(include=["object"]).columns:
    if column not in ["Churn", "customerID"]:
        df_telco[column] = label_encoder.fit_transform(df_telco[column])

df_telco["Churn"] = label_encoder.fit_transform(df_telco["Churn"])

scaler = StandardScaler()
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]
df_telco[numerical_features] = scaler.fit_transform(df_telco[numerical_features])

X = df_telco.drop(columns=["Churn", "customerID"])
y = df_telco["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

/tmp/ipykernel_535/4168804381.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_telco.select_dtypes(include=["object"]).columns:


In [16]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
accuracy_initial = accuracy_score(y_test, y_pred)

print(f"Initial (untuned) Model Accuracy: {accuracy_initial:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Initial (untuned) Model Accuracy: 0.7935

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.86      1035
           1       0.64      0.51      0.57       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.78      0.79      0.79      1409



In [17]:
param_dist = {
    "n_estimators": np.arange(50, 200, 10),
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4],
}
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist, n_iter=20, cv=5, scoring="accuracy", n_jobs=-1, random_state=42,
)
random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print(f"Best Parameters (RandomizedSearchCV): {best_params}")

best_model = random_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"\nTuned Model Accuracy: {accuracy_tuned:.4f}")
print(f"Improvement over untuned baseline: {accuracy_tuned - accuracy_initial:+.4f}")
print("\nClassification Report (Tuned Model):\n", classification_report(y_test, y_pred_tuned))

Best Parameters (RandomizedSearchCV): {'n_estimators': np.int64(50), 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 10}

Tuned Model Accuracy: 0.7935
Improvement over untuned baseline: +0.0000

Classification Report (Tuned Model):
               precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.51      0.57       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.78      0.79      0.79      1409



In [18]:
# Confirm the result with cross-validation, not just this one train/test split
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring="accuracy")

print(f"Cross-Validation Accuracy Scores: {cv_scores}")
print(f"Mean Cross-Validation Accuracy: {cv_scores.mean():.4f}")
print(f"Std Dev: {cv_scores.std():.4f}")
print("\nIf this mean is close to the single-split test accuracy above, the result is likely reliable -")
print("not just a fluke of one particular train/test split.")

Cross-Validation Accuracy Scores: [0.8105039  0.80979418 0.78566359 0.8046875  0.80397727]
Mean Cross-Validation Accuracy: 0.8029
Std Dev: 0.0090

If this mean is close to the single-split test accuracy above, the result is likely reliable -
not just a fluke of one particular train/test split.


## Week 8 Recap

| Day | Topic | Where you'll use it again |
|---|---|---|
| 1 | Manual tuning intuition | Sanity-checking automated search results |
| 2 | GridSearchCV vs RandomizedSearchCV | Every model that needs tuning |
| 3 | Optuna | Efficient tuning for expensive models (deep learning ahead) |
| 4 | Ridge/Lasso alpha tuning | Regularization strength as a hyperparameter |
| 5 | K-Fold vs Stratified K-Fold | Reliable evaluation on imbalanced data |
| 6 | Tuning workflow on a new model type (SVM) | The universal fit/search/evaluate pattern |
| 7 | Full tuned pipeline + CV confirmation | Realistic "ship this model" workflow |

**Before moving to Week 9:** you should be able to set up a `RandomizedSearchCV` or Optuna study for a new model you've never tuned before, just by knowing its hyperparameter names. This closes out the classical ML portion of the course — Week 9 begins deep learning, where many of these same ideas (train/test splits, overfitting, regularization, tuning) reappear in a new form (epochs, dropout, learning rate schedules).
